<a href="https://colab.research.google.com/github/johanndeboda/AAI2026/blob/2026fall/ML/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report


url = 'https://raw.githubusercontent.com/johanndeboda/AAI2026/refs/heads/2026fall/ML/customer_churn.csv'
df = pd.read_csv(url)

# Combine the four time-of-day columns into overall usage and spend
df['total_calls'] = df[['total day calls', 'total eve calls',
                        'total night calls', 'total intl calls']].sum(axis=1)
df['total_charge'] = df[['total day charge', 'total eve charge',
                         'total night charge', 'total intl charge']].sum(axis=1)

# Target is True/False in the file; convert to 1/0
df['churn'] = df['churn'].astype(int)

print(f"Loaded {len(df)} customers, churn rate {df['churn'].mean():.1%}")

# Features and target
X = df[['account length', 'total_calls', 'total_charge', 'customer service calls',
        'state']]
y = df['churn']
# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['account length', 'total_calls', 'total_charge',
                                   'customer service calls']),
        ('cat', OneHotEncoder(sparse_output=False), ['state'])
    ])

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

# Train model
model.fit(X_train, y_train)
# Evaluate on the held-out test set
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=['stayed', 'churned']))
# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    'account length': [128],
    'total_calls': [303],
    'total_charge': [75.56],
    'customer service calls': [5],
    'state': ['KS']
})
churn_probability = model.predict_proba(new_customer)[0][1]

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0

print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
coef_series = pd.Series(coefficients, index=feature_names)

print("\nModel Coefficients (numeric features):")
for f in ['num__account length', 'num__total_calls', 'num__total_charge',
          'num__customer service calls']:
    print(f"{f}: {coef_series[f]:.2f}")

state_coefs = coef_series[[i for i in coef_series.index
                           if i.startswith('cat__')]].sort_values()
print("\n3 lowest-risk states:")
for name, val in state_coefs.head(3).items():
    print(f"{name}: {val:.2f}")
print("\n3 highest-risk states:")
for name, val in state_coefs.tail(3).items():
    print(f"{name}: {val:.2f}")

Loaded 3333 customers, churn rate 14.5%
Accuracy: 0.855
              precision    recall  f1-score   support

      stayed       0.86      0.99      0.92       566
     churned       0.70      0.07      0.13       101

    accuracy                           0.85       667
   macro avg       0.78      0.53      0.52       667
weighted avg       0.83      0.85      0.80       667

Churn Probability for new customer: 0.73
Churn Prediction (1 = churn, 0 = no churn): 1

Model Coefficients (numeric features):
num__account length: 0.07
num__total_calls: -0.00
num__total_charge: 0.70
num__customer service calls: 0.59

3 lowest-risk states:
cat__state_VT: -0.93
cat__state_ND: -0.93
cat__state_HI: -0.72

3 highest-risk states:
cat__state_SC: 0.69
cat__state_NV: 0.69
cat__state_TX: 0.76
